# Entrenamiento de Huellas-GAN en Colab

Este notebook corre el entrenamiento de la cDCGAN condicional (Fase 4 del proyecto).

**Antes de correrlo:**

1. Subi la carpeta `huellas-gan/` completa a tu Google Drive, en `MyDrive/huellas-gan/`.
   - NO subas `venv/` (pesa mucho, no sirve en Colab).
   - SI subis `data/processed/` con `images.npz` y `metadata.csv` (ya etiquetado).
2. En Colab: *Entorno de ejecucion → Cambiar tipo de entorno → GPU (T4)*.
3. Correr las celdas de arriba hacia abajo.

Los checkpoints y samples quedan guardados en Drive (en el mismo folder), asi que si se corta la sesion de Colab no se pierde nada.

## 1. Verificar que hay GPU

In [ ]:
import torch

print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Mem total:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('OJO: no hay GPU. Ir a Entorno de ejecucion -> Cambiar tipo de entorno -> GPU.')

## 2. Montar Google Drive y moverse al repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/huellas-gan
!ls

## 3. Instalar dependencias

Colab ya trae `torch`, `numpy`, `matplotlib`, `opencv-python` y `tqdm`. Solo instalamos lo que falta (FastAPI no hace falta acá, pero el `requirements.txt` completo no molesta).

In [ ]:
!pip install -q -r requirements.txt

## 4. Smoke test (opcional, 2 iteraciones)

Chequea que los imports y el dataset esten OK antes de lanzar el entrenamiento largo.

In [ ]:
from src.training.config import TrainConfig
from src.training.train import train

train(TrainConfig(epochs=1, batch_size=16, num_workers=0,
                  max_steps=2, sample_every_epochs=1, ckpt_every_epochs=1))

## 5. Entrenamiento completo

En una T4, 50 epocas con batch 64 tardan ~30-40 minutos. Si se corta la sesion, los checkpoints en Drive permiten retomar (tenemos que cargar el `.pt` a mano; esto queda para una version siguiente del loop).

In [ ]:
cfg = TrainConfig(
    epochs=50,
    batch_size=64,
    num_workers=2,
    sample_every_epochs=2,
    ckpt_every_epochs=10,
)
train(cfg)

## 6. Ver samples y curva de perdida

In [ ]:
from IPython.display import Image, display
from pathlib import Path

samples = sorted(Path('outputs/training_samples').glob('epoch_*.png'))
print(f'{len(samples)} grillas de samples')
if samples:
    display(Image(filename=str(samples[-1])))

In [ ]:
import csv
import matplotlib.pyplot as plt

with open('outputs/training_samples/train_log.csv') as f:
    rows = list(csv.DictReader(f))
epochs = [int(r['epoch']) for r in rows]
loss_d = [float(r['loss_d']) for r in rows]
loss_g = [float(r['loss_g']) for r in rows]

plt.figure(figsize=(8, 4))
plt.plot(epochs, loss_d, label='D')
plt.plot(epochs, loss_g, label='G')
plt.xlabel('epoca')
plt.ylabel('loss (media de la epoca)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. Listo

El modelo final quedo guardado en `models/final/generator.pt` (en Drive). En Fase 5 lo cargamos para evaluar, y en Fase 6 lo usa la app web para generar huellas on-demand.

**Que bajarte a tu maquina:**

- `models/final/generator.pt` (lo usa la app)
- `outputs/training_samples/` (grilla final + log CSV — para poner en el README y ver evolucion)

`models/checkpoints/` pesa bastante (~300 MB por checkpoint x cuantos ckpt_every). Los podes dejar en Drive por si queres retomar entrenamiento mas adelante, o borrarlos si ya tenes el modelo final.